# Bronze Batch Ingestion

This notebook ingests historical batch CSV files into raw Delta tables. 
All source columns are preserved as strings, and audit metadata is added.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
RAW_DB = "retail_raw"
SILVER_DB = "retail_silver"
GOLD_DB = "retail_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {RAW_DB}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

DataFrame[]

In [0]:
BASE_PATH = "file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts"

display(dbutils.fs.ls(BASE_PATH))

path,name,size,modificationTime
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/Assignment 7 Celebal.ipynb,Assignment 7 Celebal.ipynb,15771,1782930720045
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/Sample - Superstore.csv,Sample - Superstore.csv,2287806,1782926187864
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/products_batch.csv,products_batch.csv,50712,1783622471066
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/customers_batch.csv,customers_batch.csv,147449,1783622471066
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/stores_batch.csv,stores_batch.csv,2819,1783622471077
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/orders_batch.csv,orders_batch.csv,1071308,1783622471229
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/customers_cdc_2026-04-24.csv,customers_cdc_2026-04-24.csv,15953,1783622505947
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/products_cdc_2026-04-24.csv,products_cdc_2026-04-24.csv,4755,1783622505952
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/orders_incremental_2026-04-24.csv,orders_incremental_2026-04-24.csv,204196,1783622506035
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/orders_incremental_2026-04-25.csv,orders_incremental_2026-04-25.csv,204211,1783622531484


In [0]:
orders_batch_path = f"{BASE_PATH}/orders_batch.csv"

orders_df = spark.read.option("header", True).option("inferSchema", False).csv(orders_batch_path)

display(orders_df.limit(5))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled


In [0]:
from pyspark.sql.functions import *

bronze_orders = (
    orders_df
    .withColumn("source_file", lit("orders_batch.csv"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("batch"))
)

In [0]:
display(bronze_orders.limit(5))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,source_file,ingestion_ts,load_type
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,orders_batch.csv,2026-07-09T19:08:27.500Z,batch
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,orders_batch.csv,2026-07-09T19:08:27.500Z,batch
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,orders_batch.csv,2026-07-09T19:08:27.500Z,batch
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,orders_batch.csv,2026-07-09T19:08:27.500Z,batch
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,orders_batch.csv,2026-07-09T19:08:27.500Z,batch


In [0]:
bronze_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_orders")

In [0]:
display(spark.sql("SELECT * FROM retail_raw.bronze_orders LIMIT 5"))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,source_file,ingestion_ts,load_type
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,orders_batch.csv,2026-07-09T19:08:32.990Z,batch


In [0]:
spark.sql("SELECT COUNT(*) AS total_orders FROM retail_raw.bronze_orders").show()

+------------+
|total_orders|
+------------+
|       12180|
+------------+



In [0]:
customers_batch_path = f"{BASE_PATH}/customers_batch.csv"

customers_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(customers_batch_path)

bronze_customers = (
    customers_df
    .withColumn("source_file", lit("customers_batch.csv"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("batch"))
)

bronze_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_customers")

spark.sql("SELECT COUNT(*) AS total_customers FROM retail_raw.bronze_customers").show()

+---------------+
|total_customers|
+---------------+
|           2560|
+---------------+



In [0]:
products_batch_path = f"{BASE_PATH}/products_batch.csv"

products_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(products_batch_path)

bronze_products = (
    products_df
    .withColumn("source_file", lit("products_batch.csv"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("batch"))
)

bronze_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_products")

spark.sql("SELECT COUNT(*) AS total_products FROM retail_raw.bronze_products").show()

+--------------+
|total_products|
+--------------+
|           830|
+--------------+



In [0]:
stores_batch_path = f"{BASE_PATH}/stores_batch.csv"

stores_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(stores_batch_path)

bronze_stores = (
    stores_df
    .withColumn("source_file", lit("stores_batch.csv"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("batch"))
)

bronze_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_stores")

spark.sql("SELECT COUNT(*) AS total_stores FROM retail_raw.bronze_stores").show()

+------------+
|total_stores|
+------------+
|          80|
+------------+



In [0]:
spark.sql("""
SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM retail_raw.bronze_orders
UNION ALL
SELECT 'bronze_customers', COUNT(*) FROM retail_raw.bronze_customers
UNION ALL
SELECT 'bronze_products', COUNT(*) FROM retail_raw.bronze_products
UNION ALL
SELECT 'bronze_stores', COUNT(*) FROM retail_raw.bronze_stores
""").show()

+----------------+---------+
|      table_name|row_count|
+----------------+---------+
|   bronze_orders|    12180|
|bronze_customers|     2560|
| bronze_products|      830|
|   bronze_stores|       80|
+----------------+---------+

